Author: Kira Buttrey, kbuttrey@berkeley.edu
Fall 2024

Flow cytometry analysis, developed using FlowKit python package for data handling and visualization.

Best opened as a Jupyter notebook for sequential analysis and data visualizations

In [1]:
#imports
import flowkit as fk
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
from matplotlib.widgets import PolygonSelector
from matplotlib.colors import LogNorm
import numpy as np
import pandas as pd
import seaborn as sns
import os
import math
from matplotlib import rcParams

#additional plotting functions
from sklearn.mixture import GaussianMixture
from scipy.stats import bootstrap
from scipy.stats import poisson
from scipy.optimize import brentq
from scipy.interpolate import UnivariateSpline
from scipy.optimize import curve_fit


#interactive plotting tools
# import bokeh
# from bokeh.plotting import figure, show
# from bokeh.layouts import row
# from bokeh.io import export_png

#start interactive Jupyter notebook. Seems that only one of a Bokeh, Matplotlib notebooks can be initialized in a single file
# bokeh.io.output_notebook()

In [2]:
#define color palatte for notebook 
import matplotlib.colors as mcolors
color_palette = ['#005AB5', '#DC3220', '#D55E00', '#CC79A7', '#009E73', '#56B4E9', '#E69F00', '#000000']
 
#set default color cycle for all plots
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=color_palette)
plt.rcParams['patch.facecolor'] = color_palette[0] #default for patch-based plots
plt.rcParams['lines.color'] = color_palette[1] #default for lines

Read in and explore data

In [3]:
import glob

samplenames = [str.join('_', file.split('_')[3:]).split('.')[0] for file in glob.glob('*.fcs')]
samplefiles = {str.join('_', file.split('_')[3:]).split('.')[0] : file for file in glob.glob('*.fcs')}

In [4]:
#read in the .fcs files, stored in the same folder as this code file
# Sample1_Name = fk.Sample('___FILE1_NAME_HERE____.fcs', subsample = 2000000) #subsample means # cells to analyze--this is set high to take in everything
# Sample2_Name = fk.Sample('___FILE2_NAME_HERE____.fcs', subsample = 2000000) 
# Sample3_Name = fk.Sample('___FILE3_NAME_HERE____.fcs', subsample = 2000000) 

all_experiments = { d : fk.Sample(samplefiles[d], subsample = 2000000) for d in samplefiles }
stained = { str.join('_', d.split('_')[:-1]) : all_experiments[d] for d in all_experiments if 'stained' in d }
unstained = { str.join('_', d.split('_')[:-1]) : all_experiments[d] for d in all_experiments if 'unstained' in d }

#organize sample names into lists for easy recall later. Could make addition lists based on condition, etc.
# all_experiments = [Sample1_Name, Sample2_Name, Sample3_Name]

In [5]:
all_experiments.keys()

dict_keys(['a4_dCyto_+_unstained', 'gC3_FL_-_unstained', 'gC3_FL_+_EC6_a4_unstained', 'aC2_FL_+_stained', 'a4_dCyto_+_stained', 'gC3_FL_+_stained', 'aC2_dCyto_-_unstained', 'untransfected_unstained', 'gC3_FL_-_stained', 'a4_FL_-_unstained', 'a4_dCyto_+_EC6_gC3_stained', 'aC2_dCyto_+_stained', 'gC3_dCyto_+_unstained', 'gC3_dCyto_+_EC6_a4_unstained', 'a4_dCyto_+_EC56_gC3_stained', 'a4_dCyto_+_EC56_gC3_unstained', 'a4_dCyto_+_EC6_gC3_unstained', 'gC3_FL_+_EC6_a4_stained', 'gC3_FL_+_unstained', 'aC2_dCyto_-_stained', 'gC3_dCyto_+_EC6_a4_stained', 'aC2_dCyto_+_unstained', 'gC3_dCyto_+_stained', 'a4_FL_-_stained'])

In [6]:
stained.keys()

dict_keys(['a4_dCyto_+', 'gC3_FL_-', 'gC3_FL_+_EC6_a4', 'aC2_FL_+', 'gC3_FL_+', 'aC2_dCyto_-', 'untransfected', 'a4_FL_-', 'a4_dCyto_+_EC6_gC3', 'aC2_dCyto_+', 'gC3_dCyto_+', 'gC3_dCyto_+_EC6_a4', 'a4_dCyto_+_EC56_gC3'])

In [7]:
unstained.keys()

dict_keys(['a4_dCyto_+', 'gC3_FL_-', 'gC3_FL_+_EC6_a4', 'aC2_dCyto_-', 'untransfected', 'a4_FL_-', 'gC3_dCyto_+', 'gC3_dCyto_+_EC6_a4', 'a4_dCyto_+_EC56_gC3', 'a4_dCyto_+_EC6_gC3', 'gC3_FL_+', 'aC2_dCyto_+'])

In [8]:
all_experiments[samplenames[0]]

Sample(v3.1, E3.fcs, 51 channels, 80794 events)

In [9]:
##check metadata
test = stained['gC3_FL_+']
print(test.get_metadata())
print(test.channels)
print(test.pnn_labels)

{'par': '51', 'tot': '55022', 'mode': 'L', 'datatype': 'F', 'byteord': '1,2,3,4', 'p1f': 'NA', 'p1l': 'NA', 'p1s': 'Event', 'p1v': 'NA', 'p1n': 'Event', 'p1b': '32', 'p1r': '67108864', 'p1e': '0,0', 'p2f': 'NA', 'p2l': 'NA', 'p2s': 'Time', 'p2v': 'NA', 'p2n': 'Time', 'p2b': '32', 'p2r': '67108864', 'p2e': '0,0', 'p3f': 'NA', 'p3l': '488', 'p3s': 'FSC-A', 'p3v': '100', 'p3n': 'FSC-A', 'p3b': '32', 'p3r': '1048576', 'p3e': '0,0', 'p4f': 'NA', 'p4l': '488', 'p4s': 'SSC-A', 'p4v': '250', 'p4n': 'SSC-A', 'p4b': '32', 'p4r': '1048576', 'p4e': '0,0', 'p5f': 'NA', 'p5l': '488', 'p5s': 'BL1-A', 'p5v': '350', 'p5n': 'BL1-A', 'p5b': '32', 'p5r': '1048576', 'p5e': '0,0', 'p6f': 'NA', 'p6l': '488', 'p6s': 'BL2-A', 'p6v': '462', 'p6n': 'BL2-A', 'p6b': '32', 'p6r': '1048576', 'p6e': '0,0', 'p7f': 'NA', 'p7l': '561', 'p7s': 'YL1-A', 'p7v': '300', 'p7n': 'YL1-A', 'p7b': '32', 'p7r': '1048576', 'p7e': '0,0', 'p8f': 'NA', 'p8l': '561', 'p8s': 'YL2-A', 'p8v': '376', 'p8n': 'YL2-A', 'p8b': '32', 'p8r': '10

In [10]:
#check cell counts in each file
for experiment in all_experiments:
    print(experiment, '\t\t\t', all_experiments[experiment].event_count)

a4_dCyto_+_unstained 			 80794
gC3_FL_-_unstained 			 102247
gC3_FL_+_EC6_a4_unstained 			 62410
aC2_FL_+_stained 			 40603
a4_dCyto_+_stained 			 71193
gC3_FL_+_stained 			 55729
aC2_dCyto_-_unstained 			 74435
untransfected_unstained 			 82252
gC3_FL_-_stained 			 85114
a4_FL_-_unstained 			 90172
a4_dCyto_+_EC6_gC3_stained 			 97413
aC2_dCyto_+_stained 			 94636
gC3_dCyto_+_unstained 			 82918
gC3_dCyto_+_EC6_a4_unstained 			 56759
a4_dCyto_+_EC56_gC3_stained 			 94112
a4_dCyto_+_EC56_gC3_unstained 			 95834
a4_dCyto_+_EC6_gC3_unstained 			 95302
gC3_FL_+_EC6_a4_stained 			 56925
gC3_FL_+_unstained 			 55022
aC2_dCyto_-_stained 			 71612
gC3_dCyto_+_EC6_a4_stained 			 62061
aC2_dCyto_+_unstained 			 90439
gC3_dCyto_+_stained 			 84553
a4_FL_-_stained 			 82986


Plot basic histogram comparisons to sanity check the data before heavier analysis

In [12]:
#Specify channel of interest to plot on x-axis of histogram ('FSC-A' = Forward scatter)
channel = 'SSC-A' 

construct = 'a4_FL_-'

Sample1_Name = all_experiments[construct+'_stained']
Sample2_Name = all_experiments[construct+'_unstained']

#retrieve data
Sample1_Name_channel = Sample1_Name.as_dataframe(source='raw')[channel] 
Sample2_Name_channel = Sample2_Name.as_dataframe(source='raw')[channel] 

# Combine the data to determine the range of values
combined_data = np.concatenate((Sample1_Name_channel, Sample2_Name_channel))
num_bins = int(math.sqrt(combined_data.size / 2))

#generate histogram
bins = np.linspace(combined_data.min(), combined_data.max(), num_bins)
plt.hist(Sample1_Name_channel, bins, alpha=0.5, label="Sample 1")
plt.hist(Sample2_Name_channel, bins, alpha=0.5, label="Sample 2")

#formatting
plt.gca().ticklabel_format(axis='x', style='sci', scilimits=(0, 0))
plt.legend(loc='upper right')
plt.title('Histogram Comparison') 
plt.xlabel(channel)
plt.ylabel('Event Count')
plt.show()

Drawing gates by hand, like is usually done in flow cytometry experiments. 

In [13]:
def draw_gates(sample, x_channel, y_channel, x_scale='linear', y_scale='linear', title='Gate'):
    # Create a figure and a 2x2 grid of subplots
    fig = plt.figure(figsize=(8, 8))
    gs = fig.add_gridspec(2, 2, width_ratios=[1, 0.2], height_ratios=[0.2, 1], wspace=0.05, hspace=0.05)

    # Create the main scatterplot axes
    ax_main = fig.add_subplot(gs[1, 0])

    # Create the histogram axes, sharing x and y axes with the main axes
    ax_top = fig.add_subplot(gs[0, 0], sharex=ax_main)
    ax_right = fig.add_subplot(gs[1, 1], sharey=ax_main)

    # Plot the main scatterplot
    ax_main.scatter(sample.as_dataframe(source='raw')[x_channel], 
                    sample.as_dataframe(source='raw')[y_channel], 
                    s=1, alpha=0.5)

    # Set labels and scales for the main axes
    ax_main.set_xlabel(x_channel)
    ax_main.set_ylabel(y_channel)
    ax_main.set_xscale(x_scale)
    ax_main.set_yscale(y_scale)

    #get x-axis and y-axis limits
    xmin, xmax, ymin, ymax = ax_main.axis()

    # Calculate histogram bins based on the selected scale
    if x_scale == 'log':
        x_bins = np.logspace(np.log10(xmin), np.log10(xmax), num=100)
    else:
        x_bins = np.linspace(xmin, xmax, num=100)

    if y_scale == 'log':
        y_bins = np.logspace(np.log10(ymin), np.log10(ymax), num=100)
    else:
        y_bins = np.linspace(ymin, ymax, num=100)

    # Plot the x-axis histogram
    ax_top.hist(sample.as_dataframe(source='raw')[x_channel], bins=x_bins, alpha=0.5)
    ax_top.set_ylabel('Count')
    ax_top.tick_params(axis='both', which='both', labelbottom=False, labelleft=False, bottom=False, left=False)

    # Plot the y-axis histogram
    ax_right.hist(sample.as_dataframe(source='raw')[y_channel], bins=y_bins, orientation='horizontal', alpha=0.5)
    ax_right.set_xlabel('Count')
    ax_right.tick_params(axis='both', which='both', labelbottom=False, labelleft=False, bottom=False, left=False)

    # Set the title above the top histogram
    ax_top.set_title(title)

    # Function to handle polygon selection
    global selected_verts
    selected_verts = None
    def onselect(verts):
        global selected_verts
        selected_verts = verts
        print("Selected polygon coordinates:")
        print(verts)
        
    # Call PolygonSelector
    poly1 = PolygonSelector(ax_main, onselect)

    # Show the plot
    plt.tight_layout()
    plt.show()
    
    return selected_verts


Gates are currently done using polygons, but could also be drawn on histograms using another function from FlowKit (or, plotting the channel of interest against FSC and using a polygon could approximate the histogram gates)

1. Gate on FSC-A and SSC-A to exclude all dead cells. Gates only need to be drawn for one sample and can be applied to many or all.


In [15]:
#instantiate gating strategy
g_strat = fk.GatingStrategy()

Sample1_Name = all_experiments['untransfected_unstained']

# gate1_verts = draw_gates(Sample1_Name, 'FSC-A', 'SSC-A', x_scale='linear', y_scale='linear', title='Sample 1, Gate 1')
gate1_verts = [(296081.5905714522, 4563.09234875095), (559916.4596241417, 9509.77629284648), (732922.9311341022, 17320.32988878678), (836726.8140400783, 27994.753136571868), (884303.5937053175, 44396.91568804651), (912417.1453256859, 58455.91216073906), (875653.2701298194, 64183.651464428614), (799962.9388442119, 64444.00325095996), (702646.7986198592, 61319.78181258383), (332845.4657673188, 25130.883484727085), (298244.17146532674, 16278.922742661409)]
# gate1_verts = [(400000, 6000), (1000000, 6000), (1000000, 60000), (400000, 60000)]

In [16]:
#instantiate gate
dim_a = fk.Dimension('FSC-A') 
dim_b = fk.Dimension('SSC-A') 
poly_gate = fk.gates.PolygonGate('poly1', dimensions = [dim_a, dim_b], vertices = gate1_verts) 

#add gate to strategy on root
g_strat.add_gate(poly_gate, gate_path=('root',))

In [17]:
from bokeh.models import Range1d

In [18]:
#check gating strategy on relevant experiments
smpl = []
report = []
for e in all_experiments:
    res = g_strat.gate_sample(all_experiments[e])
    # print(res.report)
    smpl.append(e)
    report.append(res.report)

    membership = res.get_gate_membership('poly1') #returns a Boolean specifying which Sample events are inside gate
    
    #plot gated cells
    # p = all_experiments[e].plot_scatter('FSC-A', 'SSC-A', source='raw', highlight_mask = membership)
    # p.y_range = Range1d(start = 0, end = 100000)
    # p.title = e
    # show(p)

In [21]:
g_strat

GatingStrategy(1 gates, 0 transforms, 0 compensations)

In [24]:
G1 = all_experiments['untransfected_unstained'].as_dataframe(source='raw', col_multi_index=False).loc[g_strat.gate_sample(all_experiments['untransfected_unstained']).get_gate_membership('poly1')]

,Event,Time,FSC-A,SSC-A,BL1-A,BL2-A,YL1-A,YL2-A,YL3-A,RL1-A,...,RL1-W,RL2-W,RL3-W,VL1-W,VL2-W,VL3-W,VL4-W,VL5-W,VL6-W,ImageFlag
0,1.0,0.343,710164.0,37348.0,2631.0,2590.0,81.0,552.0,716.0,43.0,...,0.0,0.0,0.0,13.0,49.0,35.0,32.0,29.0,32.0,1.0
1,2.0,0.343,509880.0,16480.0,1999.0,1019.0,35.0,93.0,380.0,162.0,...,0.0,0.0,0.0,0.0,37.0,20.0,0.0,10.0,0.0,1.0
2,3.0,0.346,553285.0,29477.0,2873.0,2805.0,177.0,449.0,619.0,271.0,...,0.0,0.0,0.0,7.0,45.0,41.0,10.0,30.0,33.0,1.0
3,4.0,0.347,687009.0,25856.0,2329.0,2276.0,-64.0,341.0,280.0,186.0,...,0.0,0.0,0.0,0.0,46.0,32.0,30.0,28.0,17.0,1.0
4,5.0,0.348,596249.0,21986.0,2101.0,2100.0,50.0,252.0,384.0,-6.0,...,0.0,0.0,0.0,0.0,43.0,25.0,11.0,15.0,9.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82246,82247.0,44.978,760558.0,51864.0,3034.0,1327.0,171.0,855.0,1796.0,188.0,...,0.0,0.0,0.0,27.0,56.0,58.0,37.0,42.0,31.0,0.0
82247,82248.0,44.978,685649.0,37729.0,1437.0,2941.0,116.0,242.0,-356.0,-92.0,...,0.0,0.0,0.0,18.0,52.0,44.0,6.0,29.0,24.0,0.0
82249,82250.0,44.979,375508.0,12811.0,-35.0,-651.0,-3.0,192.0,604.0,-111.0,...,0.0,0.0,0.0,0.0,38.0,16.0,0.0,10.0,0.0,0.0
82250,82251.0,44.979,502531.0,23867.0,1601.0,1540.0,21.0,-188.0,-21.0,-260.0,...,0.0,0.0,0.0,16.0,48.0,36.0,23.0,41.0,7.0,0.0


In [19]:
viability = pd.concat(report, ignore_index=True)[['sample','count']]
viability['sample'] = [each.split('.')[0] for each in viability['sample']]
viability = viability.set_index('sample')
viability

,count
sample,
E3,62209
F3,79806
H1,40573
H6,13401
E4,55258
E2,40651
G5,54912
H5,74863
F4,67428


In [18]:
wellmatching = pd.Series(smpl, index = [each.split('.')[0] for each in viability.index], name = 'sample').loc[[letter+str(n) for letter in ['E','F','G','H'] for n in np.arange(0,6,1)+1]]
wellmatching = wellmatching.to_dict()
wellmatching

{'E1': 'gC3_FL_+_unstained',
 'E2': 'gC3_FL_+_stained',
 'E3': 'a4_dCyto_+_unstained',
 'E4': 'a4_dCyto_+_stained',
 'E5': 'aC2_dCyto_+_unstained',
 'E6': 'aC2_dCyto_+_stained',
 'F1': 'gC3_dCyto_+_unstained',
 'F2': 'gC3_dCyto_+_stained',
 'F3': 'gC3_FL_-_unstained',
 'F4': 'gC3_FL_-_stained',
 'F5': 'a4_FL_-_unstained',
 'F6': 'a4_FL_-_stained',
 'G1': 'a4_dCyto_+_EC56_gC3_unstained',
 'G2': 'a4_dCyto_+_EC56_gC3_stained',
 'G3': 'a4_dCyto_+_EC6_gC3_unstained',
 'G4': 'a4_dCyto_+_EC6_gC3_stained',
 'G5': 'aC2_dCyto_-_unstained',
 'G6': 'aC2_dCyto_-_stained',
 'H1': 'gC3_FL_+_EC6_a4_unstained',
 'H2': 'gC3_FL_+_EC6_a4_stained',
 'H3': 'gC3_dCyto_+_EC6_a4_unstained',
 'H4': 'gC3_dCyto_+_EC6_a4_stained',
 'H5': 'untransfected_unstained',
 'H6': 'aC2_FL_+_stained'}

In [19]:
def decode(input):
    
    dict1 = {'E1': 'gC3_FL_+_unstained',
             'E2': 'gC3_FL_+_stained',
             'E3': 'a4_dCyto_+_unstained',
             'E4': 'a4_dCyto_+_stained',
             'E5': 'aC2_dCyto_+_unstained',
             'E6': 'aC2_dCyto_+_stained',
             'F1': 'gC3_dCyto_+_unstained',
             'F2': 'gC3_dCyto_+_stained',
             'F3': 'gC3_FL_-_unstained',
             'F4': 'gC3_FL_-_stained',
             'F5': 'a4_FL_-_unstained',
             'F6': 'a4_FL_-_stained',
             'G1': 'a4_dCyto_+_EC56_gC3_unstained',
             'G2': 'a4_dCyto_+_EC56_gC3_stained',
             'G3': 'a4_dCyto_+_EC6_gC3_unstained',
             'G4': 'a4_dCyto_+_EC6_gC3_stained',
             'G5': 'aC2_dCyto_-_unstained',
             'G6': 'aC2_dCyto_-_stained',
             'H1': 'gC3_FL_+_EC6_a4_unstained',
             'H2': 'gC3_FL_+_EC6_a4_stained',
             'H3': 'gC3_dCyto_+_EC6_a4_unstained',
             'H4': 'gC3_dCyto_+_EC6_a4_stained',
             'H5': 'untransfected_unstained',
             'H6': 'aC2_FL_+_stained'}
    
    dict2 = { v : k for k, v in dict1.items() }

    if input in dict1.keys():
        output = dict1[input]
    elif input in dict2.keys():
        output = dict2[input]

    return output

In [20]:
def multicode(input):

    dict1 = {'E1': 'gC3_FL_+_unstained',
         'E2': 'gC3_FL_+_stained',
         'E3': 'a4_dCyto_+_unstained',
         'E4': 'a4_dCyto_+_stained',
         'E5': 'aC2_dCyto_+_unstained',
         'E6': 'aC2_dCyto_+_stained',
         'F1': 'gC3_dCyto_+_unstained',
         'F2': 'gC3_dCyto_+_stained',
         'F3': 'gC3_FL_-_unstained',
         'F4': 'gC3_FL_-_stained',
         'F5': 'a4_FL_-_unstained',
         'F6': 'a4_FL_-_stained',
         'G1': 'a4_dCyto_+_EC56_gC3_unstained',
         'G2': 'a4_dCyto_+_EC56_gC3_stained',
         'G3': 'a4_dCyto_+_EC6_gC3_unstained',
         'G4': 'a4_dCyto_+_EC6_gC3_stained',
         'G5': 'aC2_dCyto_-_unstained',
         'G6': 'aC2_dCyto_-_stained',
         'H1': 'gC3_FL_+_EC6_a4_unstained',
         'H2': 'gC3_FL_+_EC6_a4_stained',
         'H3': 'gC3_dCyto_+_EC6_a4_unstained',
         'H4': 'gC3_dCyto_+_EC6_a4_stained',
         'H5': 'untransfected_unstained',
         'H6': 'aC2_FL_+_stained'}
    
    dict2 = { v : k for k, v in dict1.items() }

    output = []
    for each in input:
        if each in dict1.keys():
            output.append(dict1[each])
        elif each in dict2.keys():
            output.append(dict2[each])
        else:
            output.append(each)
    
    return output

In [21]:
celldata = pd.DataFrame([(sample, all_experiments[sample].event_count) for sample in samplenames], columns = ['sample', 'event_count'])
wellindex = viability.loc[multicode(celldata['sample'])].index
wellindex.name = 'samplewell'
celldata = celldata.set_index(wellindex)
celldata['Viable_cellcount'] = viability
celldata['Viable_fraction'] = celldata['Viable_cellcount']/celldata['event_count']
celldata

,sample,event_count,Viable_cellcount,Viable_fraction
samplewell,,,,
E3,a4_dCyto_+_unstained,80794,61980,0.767136
F3,gC3_FL_-_unstained,102247,79465,0.777187
H1,gC3_FL_+_EC6_a4_unstained,62410,40434,0.647877
H6,aC2_FL_+_stained,40603,13904,0.342438
E4,a4_dCyto_+_stained,71193,54972,0.772155
E2,gC3_FL_+_stained,55729,40502,0.726767
G5,aC2_dCyto_-_unstained,74435,54527,0.732545
H5,untransfected_unstained,82252,74643,0.907492
F4,gC3_FL_-_stained,85114,67140,0.788824


In [22]:
# gate2_verts = draw_gates(Sample1_Name, 'SSC-A', 'SSC-H', title = 'Sample 1, Gate 2')
gate2_verts = [(5083.767713185756, 8290.132375174038), (24961.477558443636, 26216.854236252802), (45943.50461732695, 40462.195715145754), (68582.00749664842, 44623.75614718189), (76588.30729543285, 41422.555814846404), (77140.46590224556, 37741.17543266058), (64716.89724895939, 31338.77476798959), (11433.59169153202, 4768.812009604993)]

2. Gate on SSC-A and SSC-H to select for single cells 

In [23]:
#add gate 2 to gating strategy in sequence with the first gate
dim_a = fk.Dimension('SSC-A') 
dim_b = fk.Dimension('SSC-H') 

poly_gate2 = fk.gates.PolygonGate('poly2', dimensions = [dim_a, dim_b], vertices = gate2_verts) 

g_strat.add_gate(poly_gate2, gate_path=('root', 'poly1'))

In [24]:
#check gating strategy on relevant experiments, if desired
smpl2 = []
report2 = []
for e in all_experiments:
    res = g_strat.gate_sample(all_experiments[e])
    # print(res.report)
    smpl2.append(e)
    report2.append(res.report)

    membership = res.get_gate_membership('poly2') #returns a Boolean specifying which Sample events are inside gate
    
    #plot gated cells
    # p = all_experiments[e].plot_scatter('SSC-A', 'SSC-H', source='raw', highlight_mask = membership) #could change highlight_mask to event_mask to exclude non-highlighted cells
    # show(p)

In [26]:
singlecells = pd.concat([df.loc[1] for df in report2], axis = 1).T[['sample','count']]
singlecells['sample'] = [each.split('.')[0] for each in singlecells['sample']]
singlecells = singlecells.set_index('sample')

celldata['Single_cells'] = singlecells
celldata['sc_fraction-of-viable'] = celldata['Single_cells']/celldata['Viable_cellcount']
celldata['sc_fraction-of-events'] = celldata['Single_cells']/celldata['event_count']
celldata

,sample,event_count,Viable_cellcount,Viable_fraction,Single_cells,sc_fraction-of-viable,sc_fraction-of-events
samplewell,,,,,,,
E3,a4_dCyto_+_unstained,80794,61980,0.767136,61711,0.99566,0.763807
F3,gC3_FL_-_unstained,102247,79465,0.777187,78921,0.993154,0.771866
H1,gC3_FL_+_EC6_a4_unstained,62410,40434,0.647877,40231,0.994979,0.644624
H6,aC2_FL_+_stained,40603,13904,0.342438,13824,0.994246,0.340467
E4,a4_dCyto_+_stained,71193,54972,0.772155,54713,0.995289,0.768517
E2,gC3_FL_+_stained,55729,40502,0.726767,40341,0.996025,0.723878
G5,aC2_dCyto_-_unstained,74435,54527,0.732545,54256,0.99503,0.728904
H5,untransfected_unstained,82252,74643,0.907492,74304,0.995458,0.90337
F4,gC3_FL_-_stained,85114,67140,0.788824,66764,0.9944,0.784407


3. Additional, custom gates on channels of interest. See FlowKit documentation for how to add gates to the strategy in parallel if relevant

In [37]:
#Specify channel of interest to plot on x-axis of histogram ('FSC-A' = Forward scatter)
channel = 'YL1-H' 

construct = 'a4_FL_-'

Sample0_Name = all_experiments['untransfected_unstained']
Sample1_Name = all_experiments[construct+'_stained']
Sample2_Name = all_experiments[construct+'_unstained']

#retrieve data
Sample0_Name_channel = Sample0_Name.as_dataframe(source='raw')[channel]
Sample1_Name_channel = Sample1_Name.as_dataframe(source='raw')[channel] 
Sample2_Name_channel = Sample2_Name.as_dataframe(source='raw')[channel] 

# Combine the data to determine the range of values
combined_data = np.concatenate((Sample0_Name_channel, Sample1_Name_channel, Sample2_Name_channel))
num_bins = int(math.sqrt(combined_data.size / 3))

#generate histogram
bins = np.linspace(combined_data.min(), combined_data.max(), num_bins)
plt.hist(Sample0_Name_channel, bins, alpha = 0.5, label = "Untransfected Control")
plt.hist(Sample1_Name_channel, bins, alpha=0.5, label="Sample 1")
plt.hist(Sample2_Name_channel, bins, alpha=0.5, label="Sample 2")

# plt.hist(Sample1_Name_channel, alpha=0.5, label="Sample 1")
# plt.hist(Sample2_Name_channel, alpha=0.5, label="Sample 2")

#formatting
plt.gca().ticklabel_format(axis='x', style='sci', scilimits=(0, 0))
plt.semilogy()
plt.legend(loc='upper right')
plt.title('Histogram Comparison') 
plt.xlabel(channel)
plt.ylabel('Event Count')
plt.show()

In [39]:
all_experiments['untransfected_unstained'].as_dataframe(source = 'raw')

pnn,Event,Time,FSC-A,SSC-A,BL1-A,BL2-A,YL1-A,YL2-A,YL3-A,RL1-A,...,RL1-W,RL2-W,RL3-W,VL1-W,VL2-W,VL3-W,VL4-W,VL5-W,VL6-W,ImageFlag
pns,Event,Time,FSC-A,SSC-A,BL1-A,BL2-A,YL1-A,YL2-A,YL3-A,RL1-A,...,RL1-W,RL2-W,RL3-W,VL1-W,VL2-W,VL3-W,VL4-W,VL5-W,VL6-W,ImageFlag
0,1.0,0.343,710164.0,37348.0,2631.0,2590.0,81.0,552.0,716.0,43.0,...,0.0,0.0,0.0,13.0,49.0,35.0,32.0,29.0,32.0,1.0
1,2.0,0.343,509880.0,16480.0,1999.0,1019.0,35.0,93.0,380.0,162.0,...,0.0,0.0,0.0,0.0,37.0,20.0,0.0,10.0,0.0,1.0
2,3.0,0.346,553285.0,29477.0,2873.0,2805.0,177.0,449.0,619.0,271.0,...,0.0,0.0,0.0,7.0,45.0,41.0,10.0,30.0,33.0,1.0
3,4.0,0.347,687009.0,25856.0,2329.0,2276.0,-64.0,341.0,280.0,186.0,...,0.0,0.0,0.0,0.0,46.0,32.0,30.0,28.0,17.0,1.0
4,5.0,0.348,596249.0,21986.0,2101.0,2100.0,50.0,252.0,384.0,-6.0,...,0.0,0.0,0.0,0.0,43.0,25.0,11.0,15.0,9.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82247,82248.0,44.978,685649.0,37729.0,1437.0,2941.0,116.0,242.0,-356.0,-92.0,...,0.0,0.0,0.0,18.0,52.0,44.0,6.0,29.0,24.0,0.0
82248,82249.0,44.978,1048575.0,59861.0,2146.0,2607.0,112.0,430.0,1362.0,-126.0,...,0.0,0.0,0.0,39.0,77.0,69.0,58.0,57.0,20.0,0.0
82249,82250.0,44.979,375508.0,12811.0,-35.0,-651.0,-3.0,192.0,604.0,-111.0,...,0.0,0.0,0.0,0.0,38.0,16.0,0.0,10.0,0.0,0.0


In [ ]:
gate3_verts = draw_gates(Sample1_Name, channel1, channel2, x_scale = 'log', title = 'Sample 1, Gate 3')

In [ ]:
dim_a = fk.Dimension(channel1) 
dim_b = fk.Dimension(channel2) 

poly_gate3 = fk.gates.PolygonGate('poly3', dimensions = [dim_a, dim_b], vertices = gate3_verts) 

g_strat.add_gate(poly_gate3, gate_path=('root', 'poly1', 'poly2'))

... move on to next section when gating strategy is fully implemented

Analysis using gated data

In [22]:
def plot_histogram_comparison(file1, gate1, label1, file2, gate2, label2, channel, title, y_prob=True):
    #extract locations of gated data as Boolean array
    resFile1 = g_strat.gate_sample(file1)
    resFile2 = g_strat.gate_sample(file2)
    membershipFile1 = resFile1.get_gate_membership(gate1) #returns a Boolean specifying which Sample events are inside gate
    membershipFile2 = resFile2.get_gate_membership(gate2) 

    #extract gated data
    file1_gatedData = file1.as_dataframe(source='raw')[channel][membershipFile1]
    file2_gatedData = file2.as_dataframe(source='raw')[channel][membershipFile2]

    #generate histogram with bins evenly distributed in log space
    combined_data = np.concatenate((file1_gatedData, file2_gatedData))
    log_max = 7 #np.log10(combined_data.max())
    bin_num = int(math.sqrt(combined_data.size/2))
    log_bins = np.logspace(0, log_max, bin_num)

    if y_prob:
        # normalize the data by total count within each sample
        file1_weights = np.ones_like(file1_gatedData) / len(file1_gatedData)
        file2_weights = np.ones_like(file2_gatedData) / len(file2_gatedData)

        plt.hist(file1_gatedData, bins=log_bins, alpha=0.5, label=label1, weights=file1_weights)
        plt.hist(file2_gatedData, bins=log_bins, alpha=0.5, label=label2, weights=file2_weights)
        plt.ylabel('Probability')
    else:
        plt.hist(file1_gatedData, bins=log_bins, alpha=0.5, label=label1)
        plt.hist(file2_gatedData, bins=log_bins, alpha=0.5, label=label2)
        plt.ylabel('Event Count')

    #formatting
    plt.legend(loc='upper left')
    plt.xscale('log')
    plt.title(title) 
    plt.xlabel(channel)
    plt.show()


In [23]:
def plot_histogram_comparison_linear(file1, gate1, label1, file2, gate2, label2, channel, title, y_prob=True):
    #extract locations of gated data as Boolean array
    resFile1 = g_strat.gate_sample(file1)
    resFile2 = g_strat.gate_sample(file2)
    membershipFile1 = resFile1.get_gate_membership(gate1) #returns a Boolean specifying which Sample events are inside gate
    membershipFile2 = resFile2.get_gate_membership(gate2) 

    #extract gated data
    file1_gatedData = file1.as_dataframe(source='raw')[channel][membershipFile1]
    file2_gatedData = file2.as_dataframe(source='raw')[channel][membershipFile2]

    #generate histogram with bins evenly distributed in log space
    combined_data = np.concatenate((file1_gatedData, file2_gatedData))
    bin_num = int(math.sqrt(combined_data.size/2))
    linear_bins = np.linspace(combined_data.min(), combined_data.max(), bin_num)

    if y_prob:
        # normalize the data by total count within each sample
        file1_weights = np.ones_like(file1_gatedData) / len(file1_gatedData)
        file2_weights = np.ones_like(file2_gatedData) / len(file2_gatedData)

        plt.hist(file1_gatedData, bins=linear_bins, alpha=0.5, label=label1, weights=file1_weights)
        plt.hist(file2_gatedData, bins=linear_bins, alpha=0.5, label=label2, weights=file2_weights)
        plt.ylabel('Probability')
    else:
        plt.hist(file1_gatedData, bins=linear_bins, alpha=0.5, label=label1)
        plt.hist(file2_gatedData, bins=linear_bins, alpha=0.5, label=label2)
        plt.ylabel('Event Count')

    #formatting
    plt.legend(loc='upper right')
    plt.title(title) 
    plt.xlabel(channel)
    plt.show()


In [ ]:
#plot two samples on the same scatter histogram plot
def plot_scatter_hist_comparison(sample1, sample1_name, sample2, sample2_name, x_channel, y_channel, gate, x_scale='linear', y_scale='linear', title='Gate'):
    # Create a figure and a 2x2 grid of subplots
    fig = plt.figure(figsize=(8, 8))
    gs = fig.add_gridspec(2, 2, width_ratios=[1, 0.2], height_ratios=[0.2, 1], wspace=0.05, hspace=0.05)

    # Create the main scatterplot axes
    ax_main = fig.add_subplot(gs[1, 0])

    # Create the histogram axes, sharing x and y axes with the main axes
    ax_top = fig.add_subplot(gs[0, 0], sharex=ax_main)
    ax_right = fig.add_subplot(gs[1, 1], sharey=ax_main)

    res1 = g_strat.gate_sample(sample1)
    membership1 = res1.get_gate_membership(gate)

    res2 = g_strat.gate_sample(sample2)
    membership2 = res2.get_gate_membership(gate)

    # Plot the main scatterplot for sample1
    ax_main.scatter(sample1.as_dataframe(source='raw')[x_channel][membership1],
                    sample1.as_dataframe(source='raw')[y_channel][membership1],
                    s=1, alpha=0.3, label=sample1_name)

    # Plot the main scatterplot for sample2
    ax_main.scatter(sample2.as_dataframe(source='raw')[x_channel][membership2],
                    sample2.as_dataframe(source='raw')[y_channel][membership2],
                    s=1, alpha=0.3, label=sample2_name)
    

    # Set labels and scales for the main axes
    ax_main.set_xlabel(x_channel)
    ax_main.set_ylabel(y_channel)
    ax_main.set_xscale(x_scale)
    ax_main.set_yscale(y_scale)
    ax_main.tick_params(axis='both', which='major', labelsize=12)

    # Add a legend to the main axes
    ax_main.legend()

    # Get x-axis and y-axis limits
    xmin, xmax, ymin, ymax = ax_main.axis()

    num_bins = int(math.sqrt(sample1.as_dataframe(source='raw')[x_channel][membership1].size))

    # Calculate histogram bins based on the selected scale
    if x_scale == 'log':
        x_bins = np.logspace(np.log10(xmin), np.log10(xmax), num=num_bins)
    else:
        x_bins = np.linspace(xmin, xmax, num=num_bins)

    if y_scale == 'log':
        y_bins = np.logspace(np.log10(ymin), np.log10(ymax), num=num_bins)
    else:
        y_bins = np.linspace(ymin, ymax, num=num_bins)

    # Plot the x-axis histogram for sample1
    ax_top.hist(sample1.as_dataframe(source='raw')[x_channel][membership1], bins=x_bins, alpha=0.5)
    # Plot the x-axis histogram for sample2
    ax_top.hist(sample2.as_dataframe(source='raw')[x_channel][membership2], bins=x_bins, alpha=0.5)
    ax_top.set_ylabel('Count', fontsize=14)
    ax_top.tick_params(axis='both', which='both', labelbottom=False, labelleft=False, bottom=False, left=False)

    # Plot the y-axis histogram for sample1
    ax_right.hist(sample1.as_dataframe(source='raw')[y_channel][membership1], bins=y_bins, orientation='horizontal', alpha=0.5)
    # Plot the y-axis histogram for sample2
    ax_right.hist(sample2.as_dataframe(source='raw')[y_channel][membership2], bins=y_bins, orientation='horizontal', alpha=0.5)
    ax_right.set_xlabel('Count', fontsize=14)
    ax_right.tick_params(axis='both', which='both', labelbottom=False, labelleft=False, bottom=False, left=False)

    # Set the title above the top histogram
    ax_top.set_title(title, fontsize=16)

    # Show the plot
    plt.show()

... can edit and add more functions for more advanced data analysis and visualization